In [15]:
class NodoCancion:
    def __init__(self, tiempo, titulo):
        self.tiempo = tiempo      
        self.titulo = titulo      
        self.izq = None           
        self.der = None          

class Gplaylist:
    def __init__(self):
        self.raiz = None

    def insertar(self, tiempo, titulo):
        nuevo = NodoCancion(tiempo, titulo)
        if self.raiz is None:
            self.raiz = nuevo
            return
        
        actual = self.raiz
        while True:
            if tiempo < actual.tiempo:
                if actual.izq is None:
                    actual.izq = nuevo
                    break
                actual = actual.izq
            else:
                if actual.der is None:
                    actual.der = nuevo
                    break
                actual = actual.der

    def calcular_tiempo(self, nodo_actual):
        if nodo_actual is None:
            return 0
        return (nodo_actual.tiempo + self.calcular_tiempo(nodo_actual.izq) + self.calcular_tiempo(nodo_actual.der))

    def recomendar_cancion(self, segundos):
        if self.raiz is None:
            return None

        actual = self.raiz
        mejor_nodo = actual
        menor_dif = abs(actual.tiempo - segundos)

        while actual is not None:
            dif_actual = abs(actual.tiempo - segundos)
            if dif_actual < menor_dif:
                menor_dif = dif_actual
                mejor_nodo = actual

            if dif_actual == 0:
                break

            if segundos < actual.tiempo:
                actual = actual.izq
            else:
                actual = actual.der
        return mejor_nodo

    def canciones_cortas(self, tiempo_min):
        self.raiz = self._filtrar_recursivo(self.raiz, tiempo_min)

    def _filtrar_recursivo(self, nodo, tiempo_min):
        if nodo is None:
            return None
        nodo.izq = self._filtrar_recursivo(nodo.izq, tiempo_min)
        nodo.der = self._filtrar_recursivo(nodo.der, tiempo_min)

        if nodo.tiempo < tiempo_min:
            return nodo.der
        return nodo

    def rango(self, nodo, tiempo_min, tiempo_max):
        if nodo is None:
            return 0
        if tiempo_min <= nodo.tiempo <= tiempo_max:
            return (1 + self.rango(nodo.izq, tiempo_min, tiempo_max) 
                      + self.rango(nodo.der, tiempo_min, tiempo_max))
        if nodo.tiempo < tiempo_min:
            return self.rango(nodo.der, tiempo_min, tiempo_max)
        return self.rango(nodo.izq, tiempo_min, tiempo_max)

    def lista(self, nodo, lista_acumulada):
        if nodo:
            self.lista(nodo.izq, lista_acumulada)
            lista_acumulada.append((nodo.tiempo, nodo.titulo))
            self.lista(nodo.der, lista_acumulada)

    def _calcular_coordenadas(self, nodo, x=0, y=0, espacio=10, data=None):
        if data is None:
            data = []
        if nodo:
            data.append((x, y, f"{nodo.tiempo}s\n{nodo.titulo[:12]}"))
            if nodo.izq:
                data.append(((x, y), (x - espacio, y - 1)))
                self._calcular_coordenadas(nodo.izq, x - espacio, y - 1, espacio * 0.6, data)
            if nodo.der:
                data.append(((x, y), (x + espacio, y - 1)))
                self._calcular_coordenadas(nodo.der, x + espacio, y - 1, espacio * 0.6, data)
        return data

print("Ingresado a la memoria")




Ingresado a la memoria


In [14]:
# Celda 2: Panel de Control Interactivo con Gráfica

import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

playlist = Gplaylist()

salida_lista = widgets.Output()
salida_acciones = widgets.Output()
salida_grafica = widgets.Output()

def actualizar_vista_canciones():
    with salida_lista:
        clear_output()
        print("====== PLAYLIST EN MEMORIA (BST) ======")
        lista_canciones = []
        try:
            playlist.lista(playlist.raiz, lista_canciones)
            if not lista_canciones:
                print("[Árbol Vacío] Usa el panel para agregar canciones.")
            for tiempo, titulo in lista_canciones:
                print(f" Tiempo: {tiempo}s \t-> {titulo}")
        except Exception as e:
            print(f"Error: {e}")
        print("=======================================")

def graficar_arbol_matplotlib():
    with salida_grafica:
        clear_output()
        if playlist.raiz is None:
            print("No hay nada que graficar, el árbol está vacío.")
            return
        
        elementos = playlist._calcular_coordenadas(playlist.raiz)
        
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.axis('off')
        
        # 1. Dibujar conexiones (ramas)
        for elem in elementos:
            if len(elem) == 2:
                p1, p2 = elem
                ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color='#1DB954', zorder=1, lw=2)
                
        # 2. Dibujar nodos (círculos y títulos)
        for elem in elementos:
            if len(elem) != 2:
                x, y, texto = elem
                ax.scatter(x, y, color='#282828', s=1600, edgecolors='#1DB954', linewidths=2, zorder=2)
                ax.text(x, y, texto, color='white', ha='center', va='center', fontsize=8, fontweight='bold', zorder=3)
                
        plt.tight_layout()
        plt.show()

# --- Controladores de Eventos ---
def on_insertar_click(b):
    try:
        tmp = int(txt_tiempo.value)
        tit = txt_titulo.value.strip()
        if not tit: raise ValueError
        playlist.insertar(tmp, tit)
        txt_tiempo.value = ""
        txt_titulo.value = ""
        actualizar_vista_canciones()
        graficar_arbol_matplotlib()
        with salida_acciones:
            clear_output()
            print(f"Guardado con éxito: '{tit}' ({tmp}s)")
    except ValueError:
        with salida_acciones:
            clear_output()
            print("Error: Ingresa datos válidos.")

def on_tiempo_total_click(b):
    total = playlist.calcular_tiempo(playlist.raiz)
    with salida_acciones:
        clear_output()
        print(f"Tiempo Total: {total} segundos.")

def on_recomendar_click(b):
    try:
        seg = int(txt_buscar.value)
        nodo = playlist.recomendar_cancion(seg)
        with salida_acciones:
            clear_output()
            if nodo: print(f"Sugerencia Perfecta: {nodo.titulo} ({nodo.tiempo}s)")
            else: print("Árbol vacío.")
    except ValueError: pass

def on_filtrar_click(b):
    try:
        minimo = int(txt_filtro.value)
        playlist.canciones_cortas(minimo)
        actualizar_vista_canciones()
        graficar_arbol_matplotlib()
        with salida_acciones:
            clear_output()
            print(f"Filtro aplicado. Menores a {minimo}s eliminados.")
    except ValueError: pass

def on_rango_click(b):
    try:
        cant = playlist.rango(playlist.raiz, int(txt_rmin.value), int(txt_rmax.value))
        with salida_acciones:
            clear_output()
            print(f"Canciones en rango: {cant}")
    except ValueError: pass

def on_graficar_click(b):
    graficar_arbol_matplotlib()

# --- Construcción Gráfica de los Controles ---
txt_tiempo = widgets.Text(description="Tiempo (s):", layout=widgets.Layout(width='180px'))
txt_titulo = widgets.Text(description="Título:", layout=widgets.Layout(width='220px'))
btn_insertar = widgets.Button(description="Insertar Track", button_style='success')
btn_insertar.on_click(on_insertar_click)
box_insertar = widgets.HBox([txt_tiempo, txt_titulo, btn_insertar])

btn_tiempo = widgets.Button(description="Calcular Tiempo Total", button_style='info', layout=widgets.Layout(width='250px'))
btn_tiempo.on_click(on_tiempo_total_click)

txt_buscar = widgets.Text(description="Segundos:", layout=widgets.Layout(width='180px'))
btn_recomendar = widgets.Button(description="Sugerencia Inteligente", button_style='warning')
btn_recomendar.on_click(on_recomendar_click)
box_recomendar = widgets.HBox([txt_buscar, btn_recomendar])

txt_filtro = widgets.Text(description="Mínimo (s):", layout=widgets.Layout(width='180px'))
btn_filtrar = widgets.Button(description="Limpiar Basura", button_style='danger')
btn_filtrar.on_click(on_filtrar_click)
box_filtrar = widgets.HBox([txt_filtro, btn_filtrar])

txt_rmin = widgets.Text(description="Rango Min:", layout=widgets.Layout(width='150px'))
txt_rmax = widgets.Text(description="Rango Max:", layout=widgets.Layout(width='150px'))
btn_rango = widgets.Button(description="Analizar Rango", button_style='primary')
btn_rango.on_click(on_rango_click)
box_rango = widgets.HBox([txt_rmin, txt_rmax, btn_rango])

btn_graficar = widgets.Button(description="Dibujar Árbol (Gráfica)", layout=widgets.Layout(width='250px'))
btn_graficar.on_click(on_graficar_click)

panel_controles = widgets.VBox([
    widgets.HTML("<h3>Control de Gplaylist</h3>"),
    box_insertar, widgets.HTML("<hr>"),
    btn_tiempo, widgets.HTML("<hr>"),
    box_recomendar, widgets.HTML("<hr>"),
    box_filtrar, widgets.HTML("<hr>"),
    box_rango, widgets.HTML("<hr>"),
    btn_graficar, widgets.HTML("<hr>"),
    widgets.HTML("<h4>Terminal:</h4>"),
    salida_acciones
], layout=widgets.Layout(width='45%'))

panel_visual = widgets.VBox([
    salida_lista,
    widgets.HTML("<h4>Estructura del Árbol:</h4>"),
    salida_grafica
], layout=widgets.Layout(width='50%', margin='0 0 0 20px'))

interface = widgets.HBox([panel_controles, panel_visual])

display(interface)
actualizar_vista_canciones()